In [1]:
# =====================================
# RADFUSE v2 — FUSED EMBEDDINGS + FULL EVALUATION + PPT PLOTS
#
# Changes from v1:
#  1. Model.forward() now returns `fused` vector (image+text combined)
#  2. Embeddings file saves fused, v_emb, t_emb, labels, preds
#  3. Test-set evaluation with per-class AUC, F1, precision, recall
#  4. Publication-ready plots saved to ./radfuse_plots/:
#       - train_val_loss_curve.png
#       - auc_per_epoch.png
#       - per_class_auc_bar.png
#       - confusion_matrix_grid.png
#       - tsne_fused_embeddings.png
#       - cosine_sim_heatmap.png
#       - pr_curves.png
# =====================================

In [2]:
import os, ast, random, torch, math
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    average_precision_score, confusion_matrix, roc_curve,
    precision_recall_curve,
)
from sklearn.manifold import TSNE
from transformers import AutoTokenizer, AutoModel
from torchvision.models import vit_b_16, ViT_B_16_Weights
from tqdm import tqdm

In [3]:
# Free any GPU memory left over from a previous run
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    free  = torch.cuda.mem_get_info()[0] / 1e9
    total = torch.cuda.mem_get_info()[1] / 1e9
    print(f"GPU memory: {free:.1f} GB free / {total:.1f} GB total")

GPU: Tesla T4
GPU memory: 15.5 GB free / 15.6 GB total


In [4]:
# ── Plot style ──────────────────────────────────────────────────────────────
PLOT_DIR = "./radfuse_plots"
os.makedirs(PLOT_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor":  "#0D1117",
    "axes.facecolor":    "#161B22",
    "axes.edgecolor":    "#30363D",
    "axes.labelcolor":   "#E6EDF3",
    "xtick.color":       "#8B949E",
    "ytick.color":       "#8B949E",
    "text.color":        "#E6EDF3",
    "grid.color":        "#21262D",
    "grid.linewidth":    0.8,
    "font.family":       "DejaVu Sans",
    "font.size":         11,
    "axes.titlesize":    13,
    "axes.titleweight":  "bold",
    "figure.dpi":        150,
    "savefig.dpi":       200,
    "savefig.bbox":      "tight",
    "savefig.facecolor": "#0D1117",
})

# Two accent colours used throughout
C_TEAL   = "#00B4D8"
C_AMBER  = "#FFB703"
C_GREEN  = "#56D364"
C_RED    = "#F85149"
C_PURPLE = "#BC8CFF"

In [5]:
# =============================
# CONFIG
# =============================
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE    = 4
ACCUM_STEPS   = 4
EPOCHS        = 20
LR            = 1e-4
LR_BACKBONE   = 5e-6
TEXT_MAX_LEN  = 96
UNFREEZE_TOP  = 4

IMG_DIR   = "/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset/official_data_iccv_final"
TRAIN_CSV = "/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset/mimic_cxr_aug_train.csv"
VAL_CSV   = "/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset/mimic_cxr_aug_validate.csv"
TEST_CSV  = "/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset/mimic_cxr_aug_test.csv"

CHEX_LABELS = [
    "Enlarged Cardiomediastinum","Cardiomegaly","Lung Opacity","Lung Lesion",
    "Edema","Consolidation","Pneumonia","Atelectasis","Pneumothorax",
    "Pleural Effusion","Pleural Other","Fracture","Support Devices","No Finding"
]
N_CLASSES = len(CHEX_LABELS)

In [6]:
# =============================
# DATASET
# =============================
class MIMICDataset(Dataset):
    def __init__(self, df, tokenizer, is_train=True):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        if is_train:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.Grayscale(num_output_channels=3),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(10),
                transforms.ToTensor(),
                transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.Grayscale(num_output_channels=3),
                transforms.ToTensor(),
                transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img_entry = row["image"]
            if isinstance(img_entry, str) and img_entry.startswith("["):
                img_entry = ast.literal_eval(img_entry)[0]
            img_path = os.path.join(IMG_DIR, img_entry)
            if not os.path.exists(img_path):
                raise FileNotFoundError
            image = Image.open(img_path).convert("RGB")
            image = self.transform(image)
        except Exception:
            image = torch.zeros(3, 224, 224)

        text   = str(row.get("text", row.get("report", "")))
        tokens = self.tokenizer(
            text, padding="max_length", truncation=True,
            max_length=TEXT_MAX_LEN, return_tensors="pt"
        )
        tokens = {k: v.squeeze(0) for k, v in tokens.items()}

        label = torch.tensor(
            [1 if lbl.lower() in text.lower() else 0 for lbl in CHEX_LABELS]
        ).float()

        return image, tokens, label


# =============================
# MODEL  ─  now returns `fused`
# =============================
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        vit_full       = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        vit_full.heads = nn.Identity()
        n_vit    = len(vit_full.encoder.layers)
        freeze_v = n_vit - UNFREEZE_TOP

        self.vit_patch   = vit_full.conv_proj
        self.vit_class   = nn.Parameter(vit_full.class_token.clone(), requires_grad=False)
        self.register_buffer('vit_pos', vit_full.encoder.pos_embedding)
        self.vit_dropout = vit_full.encoder.dropout
        self.vit_frozen  = nn.Sequential(*list(vit_full.encoder.layers)[:freeze_v])
        self.vit_top     = nn.Sequential(*list(vit_full.encoder.layers)[freeze_v:])
        self.vit_norm    = vit_full.encoder.ln

        for m in [self.vit_patch, self.vit_frozen]:
            for p in m.parameters():
                p.requires_grad = False

        self.bert = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
        for p in self.bert.parameters():
            p.requires_grad = False
        self.bert = self.bert.cpu()

        self.cross_v2t = nn.MultiheadAttention(768, num_heads=8, dropout=0.1, batch_first=True)
        self.cross_t2v = nn.MultiheadAttention(768, num_heads=8, dropout=0.1, batch_first=True)
        self.norm_v    = nn.LayerNorm(768)
        self.norm_t    = nn.LayerNorm(768)

        self.proj = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.4),
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, N_CLASSES)
        )

        # ── Contrastive heads (128-d, L2-normalised) ─────────────────────
        self.proj_v   = nn.Linear(768, 128)
        self.proj_t   = nn.Linear(768, 128)
        self.log_temp = nn.Parameter(torch.tensor([-2.0]))

        # ── NEW: fused projection head (768-d → 256-d for CMKG) ──────────
        # Keeps the full semantic richness while reducing dimensionality.
        self.fused_proj = nn.Sequential(
            nn.LayerNorm(768),
            nn.Linear(768, 256),
            nn.GELU(),
        )

        self._cpu_modules = ['vit_patch', 'vit_frozen', 'bert']

    def to(self, *args, **kwargs):
        super().to(*args, **kwargs)
        for name in self._cpu_modules:
            getattr(self, name).cpu().float()
        if hasattr(self, 'vit_pos'):
            self.vit_pos = self.vit_pos.cpu().float()
        return self

    def _vit_forward(self, img):
        with torch.no_grad():
            x   = self.vit_patch(img.cpu().float())
            x   = x.flatten(2).transpose(1, 2)
            B   = x.shape[0]
            cls = self.vit_class.detach().cpu().float().expand(B, -1, -1)
            x   = torch.cat([cls, x], dim=1)
            x   = x + self.vit_pos.cpu().float()
            x   = self.vit_dropout(x)
            x   = self.vit_frozen(x)
        x = self.vit_top(x.to(DEVICE))
        x = self.vit_norm(x)
        return x   # (B, 197, 768)

    def _bert_forward(self, tokens):
        with torch.no_grad():
            out = self.bert(
                input_ids=tokens['input_ids'].cpu(),
                attention_mask=tokens['attention_mask'].cpu(),
                token_type_ids=tokens.get(
                    'token_type_ids',
                    torch.zeros_like(tokens['input_ids'])
                ).cpu(),
            )
            t_seq = out.last_hidden_state.float()
        return t_seq.to(DEVICE)   # (B, seq, 768)

    def forward(self, img, tokens):
        V = self._vit_forward(img)    # (B, 197, 768)
        T = self._bert_forward(tokens) # (B, seq,  768)

        V_ca, _ = self.cross_v2t(V, T, T)
        T_ca, _ = self.cross_t2v(T, V, V)
        V = self.norm_v(V + V_ca)
        T = self.norm_t(T + T_ca)

        v_cls = V[:, 0, :]   # (B, 768)
        t_cls = T[:, 0, :]   # (B, 768)

        # ── Fused representation (additive, then projected) ───────────────
        fused_raw = (v_cls + t_cls) / 2.0          # (B, 768) — for classification
        fused_emb = self.fused_proj(fused_raw)      # (B, 256) — for CMKG graph nodes
        fused_emb = F.normalize(fused_emb, dim=-1)  # L2-normalised → cosine-ready

        logits = self.proj(fused_raw)
        v_emb  = F.normalize(self.proj_v(v_cls), dim=-1)   # (B, 128)
        t_emb  = F.normalize(self.proj_t(t_cls), dim=-1)   # (B, 128)

        return logits, v_emb, t_emb, fused_emb

In [7]:
# =============================
# LOSS
# =============================
def info_nce(v_emb, t_emb, log_temp):
    tau    = log_temp.exp().clamp(0.07, 0.3)
    logits = (v_emb @ t_emb.T) / tau
    labels = torch.arange(v_emb.size(0)).to(DEVICE)
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

def focal_loss(logits, targets, gamma=2.0, smoothing=0.1):
    targets_s = targets * (1 - smoothing) + 0.5 * smoothing
    bce       = F.binary_cross_entropy_with_logits(logits, targets_s, reduction='none')
    p_t       = torch.sigmoid(logits) * targets + (1 - torch.sigmoid(logits)) * (1 - targets)
    return ((1 - p_t) ** gamma * bce).mean()

In [8]:
# =============================
# TRAIN
# =============================
def train_epoch(model, loader, optimizer, scaler):
    model.train()
    total = 0.0
    optimizer.zero_grad()
    for step, (img, tokens, labels) in enumerate(tqdm(loader, desc="Train")):
        img, labels = img.to(DEVICE), labels.to(DEVICE)
        with torch.amp.autocast('cuda'):
            logits, v_emb, t_emb, _ = model(img, tokens)
            loss = (focal_loss(logits, labels) +
                    0.5 * info_nce(v_emb, t_emb, model.log_temp)) / ACCUM_STEPS
        scaler.scale(loss).backward()
        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        total += loss.item() * ACCUM_STEPS
    return total / len(loader)

In [9]:
# =============================
# VALIDATE
# =============================
@torch.no_grad()
def validate(model, loader):
    model.eval()
    total, all_preds, all_labels = 0.0, [], []
    for img, tokens, labels in tqdm(loader, desc="Val  "):
        img, labels = img.to(DEVICE), labels.to(DEVICE)
        with torch.amp.autocast('cuda'):
            logits, v_emb, t_emb, _ = model(img, tokens)
            loss = focal_loss(logits, labels) + 0.5 * info_nce(v_emb, t_emb, model.log_temp)
        total += loss.item()
        all_preds.append(torch.sigmoid(logits).cpu())
        all_labels.append(labels.cpu())

    preds  = torch.cat(all_preds).numpy()
    truths = torch.cat(all_labels).numpy()
    valid  = [i for i in range(truths.shape[1])
              if truths[:,i].sum() > 0 and (1-truths[:,i]).sum() > 0]
    auc = roc_auc_score(truths[:,valid], preds[:,valid], average='macro') if valid else 0.0
    return total / len(loader), auc

In [10]:
# =============================
# FULL TEST EVALUATION
# =============================
@torch.no_grad()
def evaluate_test(model, loader):
    """
    Returns per-class and macro metrics on the test set.
    Also collects fused embeddings for downstream CMKG use.
    """
    model.eval()
    all_preds, all_labels = [], []
    all_v, all_t, all_fused = [], [], []

    for img, tokens, labels in tqdm(loader, desc="Test "):
        img = img.to(DEVICE)
        with torch.amp.autocast('cuda'):
            logits, v_emb, t_emb, fused_emb = model(img, tokens)
        all_preds.append(torch.sigmoid(logits).cpu())
        all_labels.append(labels)
        all_v.append(v_emb.cpu())
        all_t.append(t_emb.cpu())
        all_fused.append(fused_emb.cpu())

    preds  = torch.cat(all_preds).numpy()    # (N, 14)
    truths = torch.cat(all_labels).numpy()   # (N, 14)
    bin_preds = (preds >= 0.5).astype(int)

    per_class = {}
    for i, name in enumerate(CHEX_LABELS):
        pos = truths[:, i].sum()
        neg = (1 - truths[:, i]).sum()
        if pos > 0 and neg > 0:
            auc = roc_auc_score(truths[:, i], preds[:, i])
            ap  = average_precision_score(truths[:, i], preds[:, i])
        else:
            auc = ap = float('nan')
        f1  = f1_score(truths[:, i], bin_preds[:, i], zero_division=0)
        pre = precision_score(truths[:, i], bin_preds[:, i], zero_division=0)
        rec = recall_score(truths[:, i], bin_preds[:, i], zero_division=0)
        per_class[name] = {"auc": auc, "ap": ap, "f1": f1,
                           "precision": pre, "recall": rec,
                           "support": int(pos)}

    valid = [i for i in range(N_CLASSES)
             if not np.isnan(per_class[CHEX_LABELS[i]]["auc"])]
    macro_auc = np.nanmean([per_class[n]["auc"] for n in CHEX_LABELS])
    macro_f1  = f1_score(truths, bin_preds, average='macro', zero_division=0)

    embeddings = {
        "fused_emb": torch.cat(all_fused),   # ← PRIMARY: 256-d, L2-norm, CMKG-ready
        "v_emb":     torch.cat(all_v),
        "t_emb":     torch.cat(all_t),
        "labels":    torch.cat(all_labels),
        "predictions": torch.cat(all_preds),
        "chex_labels": CHEX_LABELS,
        "macro_auc":   macro_auc,
        "macro_f1":    macro_f1,
        "per_class":   per_class,
    }

    return per_class, macro_auc, macro_f1, preds, truths, embeddings

In [11]:
# =============================
# PLOTS
# =============================

def plot_loss_curves(history):
    """Train vs Val loss over epochs."""
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(epochs, history["train_loss"], color=C_TEAL,  lw=2.5, label="Train Loss", marker='o', ms=4)
    ax.plot(epochs, history["val_loss"],   color=C_AMBER, lw=2.5, label="Val Loss",   marker='s', ms=4)
    ax.fill_between(epochs, history["train_loss"], history["val_loss"],
                    alpha=0.08, color=C_PURPLE)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("Training vs Validation Loss")
    ax.legend(framealpha=0.3); ax.grid(True, alpha=0.4)
    _annotate_best(ax, history["val_loss"], epochs, C_AMBER, "Best Val")
    fig.tight_layout()
    fig.savefig(f"{PLOT_DIR}/train_val_loss_curve.png")
    plt.close(fig)
    print("  Saved: train_val_loss_curve.png")


def plot_auc_curve(history):
    """Val AUC over epochs."""
    epochs = range(1, len(history["val_auc"]) + 1)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(epochs, history["val_auc"], color=C_GREEN, lw=2.5, marker='D', ms=4)
    ax.fill_between(epochs, history["val_auc"], alpha=0.15, color=C_GREEN)
    ax.axhline(max(history["val_auc"]), color=C_RED, lw=1.2, ls='--', alpha=0.7,
               label=f"Best AUC = {max(history['val_auc']):.4f}")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Macro AUC")
    ax.set_title("Validation AUC per Epoch")
    ax.legend(framealpha=0.3); ax.grid(True, alpha=0.4)
    fig.tight_layout()
    fig.savefig(f"{PLOT_DIR}/auc_per_epoch.png")
    plt.close(fig)
    print("  Saved: auc_per_epoch.png")


def plot_per_class_auc(per_class):
    """Horizontal bar chart of per-class AUC on test set."""
    names = [n for n in CHEX_LABELS if not np.isnan(per_class[n]["auc"])]
    aucs  = [per_class[n]["auc"] for n in names]
    idx   = np.argsort(aucs)
    names_s = [names[i] for i in idx]
    aucs_s  = [aucs[i]  for i in idx]

    colors = [C_RED if a < 0.80 else C_AMBER if a < 0.90 else C_GREEN for a in aucs_s]

    fig, ax = plt.subplots(figsize=(10, 7))
    bars = ax.barh(names_s, aucs_s, color=colors, edgecolor='none', height=0.65)
    ax.axvline(0.5, color='#555', lw=1, ls='--')
    ax.axvline(0.9, color=C_GREEN, lw=1, ls='--', alpha=0.5, label='AUC = 0.90')
    for bar, val in zip(bars, aucs_s):
        ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
                f"{val:.3f}", va='center', fontsize=9.5, color='#E6EDF3')
    ax.set_xlim(0.4, 1.05)
    ax.set_xlabel("AUC (ROC)")
    ax.set_title("Per-Class AUC — Test Set")
    ax.legend(framealpha=0.3); ax.grid(True, axis='x', alpha=0.3)
    fig.tight_layout()
    fig.savefig(f"{PLOT_DIR}/per_class_auc_bar.png")
    plt.close(fig)
    print("  Saved: per_class_auc_bar.png")


def plot_confusion_matrices(truths, preds_bin):
    """3×5 grid of per-class confusion matrices (binary)."""
    fig = plt.figure(figsize=(18, 12))
    cmap = LinearSegmentedColormap.from_list("rf", ["#161B22", C_TEAL])

    for i, name in enumerate(CHEX_LABELS):
        ax = fig.add_subplot(3, 5, i + 1)

        # ✅ FIX: enforce 2x2 matrix
        cm = confusion_matrix(truths[:, i], preds_bin[:, i], labels=[0, 1])

        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap=cmap,
            ax=ax,
            cbar=False,
            linewidths=0.5,
            linecolor='#21262D',
            annot_kws={"size": 11}
        )

        ax.set_title(name, fontsize=9.5, pad=4)

        # ✅ FIX: set ticks properly
        ax.set_xticks([0.5, 1.5])
        ax.set_xticklabels(["Neg", "Pos"], fontsize=8)

        ax.set_yticks([0.5, 1.5])
        ax.set_yticklabels(["Neg", "Pos"], fontsize=8, rotation=0)

        ax.set_xlabel("Predicted", fontsize=8)
        ax.set_ylabel("True", fontsize=8)

    fig.suptitle("Per-Class Confusion Matrices — Test Set", fontsize=14, y=1.01)
    fig.tight_layout()
    fig.savefig(f"{PLOT_DIR}/confusion_matrix_grid.png")
    plt.close(fig)

    print("  Saved: confusion_matrix_grid.png")


def plot_tsne(embeddings_dict):
    """2-D t-SNE of fused embeddings, coloured by dominant pathology."""
    fused  = embeddings_dict["fused_emb"].numpy()   # (N, 256)
    labels = embeddings_dict["labels"].numpy()       # (N, 14)

    # Dominant label per sample (highest-weight class; -1 = multi/none)
    dominant = np.argmax(labels, axis=1)
    dominant[labels.max(axis=1) == 0] = N_CLASSES   # no-label bucket

    print("  Running t-SNE on fused embeddings …")
    tsne  = TSNE(n_components=2, perplexity=30, random_state=42,
                 n_iter=1000, learning_rate='auto', init='pca')
    emb2d = tsne.fit_transform(fused)

    palette = plt.cm.tab20(np.linspace(0, 1, N_CLASSES + 1))
    fig, ax = plt.subplots(figsize=(12, 9))
    for cls_idx in range(N_CLASSES + 1):
        mask = dominant == cls_idx
        if mask.sum() == 0:
            continue
        label_name = CHEX_LABELS[cls_idx] if cls_idx < N_CLASSES else "None"
        ax.scatter(emb2d[mask, 0], emb2d[mask, 1],
                   c=[palette[cls_idx]], s=18, alpha=0.75,
                   label=label_name, edgecolors='none')
    ax.legend(fontsize=8, markerscale=1.5, ncol=2,
              framealpha=0.2, loc='upper right')
    ax.set_title("t-SNE of Fused Embeddings (256-d CMKG Vectors) — Test Set")
    ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")
    ax.grid(True, alpha=0.2)
    fig.tight_layout()
    fig.savefig(f"{PLOT_DIR}/tsne_fused_embeddings.png")
    plt.close(fig)
    print("  Saved: tsne_fused_embeddings.png")


def plot_cosine_sim_heatmap(embeddings_dict, n_samples=200):
    """Cosine similarity heatmap across a random subset — validates cluster structure."""
    fused = embeddings_dict["fused_emb"].numpy()
    rng   = np.random.default_rng(42)
    idx   = rng.choice(len(fused), min(n_samples, len(fused)), replace=False)
    sub   = fused[idx]                       # already L2-normalised
    sim   = sub @ sub.T                      # cosine similarity matrix

    fig, ax = plt.subplots(figsize=(9, 8))
    cmap = LinearSegmentedColormap.from_list("sim", ["#161B22", C_PURPLE, C_TEAL, "#FFFFFF"])
    im = ax.imshow(sim, cmap=cmap, vmin=-1, vmax=1, aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Cosine Similarity")
    ax.set_title(f"Fused Embedding Cosine Similarity\n({n_samples}-sample subset, Test Set)")
    ax.set_xlabel("Sample index"); ax.set_ylabel("Sample index")
    fig.tight_layout()
    fig.savefig(f"{PLOT_DIR}/cosine_sim_heatmap.png")
    plt.close(fig)
    print("  Saved: cosine_sim_heatmap.png")


def plot_pr_curves(truths, preds, per_class):
    """Precision-Recall curves for all valid classes."""
    valid_names = [n for n in CHEX_LABELS if not np.isnan(per_class[n]["auc"])]
    palette     = plt.cm.tab20(np.linspace(0, 1, len(valid_names)))

    fig, ax = plt.subplots(figsize=(10, 8))
    for i, name in enumerate(valid_names):
        ci = CHEX_LABELS.index(name)
        prec, rec, _ = precision_recall_curve(truths[:, ci], preds[:, ci])
        ap = per_class[name]["ap"]
        ax.plot(rec, prec, lw=1.6, color=palette[i],
                label=f"{name} (AP={ap:.2f})")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curves — Test Set")
    ax.legend(fontsize=7.5, ncol=2, framealpha=0.2, loc='lower left')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(f"{PLOT_DIR}/pr_curves.png")
    plt.close(fig)
    print("  Saved: pr_curves.png")


def plot_metrics_summary(per_class, macro_auc, macro_f1):
    """Grouped bar chart: AUC, F1, Precision, Recall per class."""
    names     = [n for n in CHEX_LABELS if not np.isnan(per_class[n]["auc"])]
    short     = [n.replace(" ", "\n") for n in names]
    aucs      = [per_class[n]["auc"]       for n in names]
    f1s       = [per_class[n]["f1"]        for n in names]
    precs     = [per_class[n]["precision"] for n in names]
    recs      = [per_class[n]["recall"]    for n in names]

    x    = np.arange(len(names))
    w    = 0.2
    fig, ax = plt.subplots(figsize=(16, 6))
    ax.bar(x - 1.5*w, aucs,  w, label="AUC",       color=C_TEAL,   alpha=0.85)
    ax.bar(x - 0.5*w, f1s,   w, label="F1",        color=C_GREEN,  alpha=0.85)
    ax.bar(x + 0.5*w, precs, w, label="Precision", color=C_AMBER,  alpha=0.85)
    ax.bar(x + 1.5*w, recs,  w, label="Recall",    color=C_PURPLE, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(short, fontsize=8)
    ax.set_ylim(0, 1.12)
    ax.axhline(macro_auc, color=C_TEAL,  ls='--', lw=1.2, alpha=0.6,
               label=f"Macro AUC={macro_auc:.3f}")
    ax.axhline(macro_f1,  color=C_GREEN, ls='--', lw=1.2, alpha=0.6,
               label=f"Macro F1={macro_f1:.3f}")
    ax.set_ylabel("Score"); ax.set_title("Per-Class Metrics Summary — Test Set")
    ax.legend(fontsize=9, framealpha=0.3, ncol=6, loc='upper center',
              bbox_to_anchor=(0.5, 1.13))
    ax.grid(True, axis='y', alpha=0.3)
    fig.tight_layout()
    fig.savefig(f"{PLOT_DIR}/metrics_summary.png")
    plt.close(fig)
    print("  Saved: metrics_summary.png")


# ── helpers ──────────────────────────────────────────────────────────────────
def _annotate_best(ax, values, epochs, color, label):
    best_ep  = int(np.argmin(values)) + 1
    best_val = min(values)
    ax.annotate(f"{label}\n{best_val:.4f}",
                xy=(best_ep, best_val),
                xytext=(best_ep + 0.5, best_val + 0.003),
                fontsize=9, color=color,
                arrowprops=dict(arrowstyle='->', color=color, lw=1.4))

In [12]:
# =============================
# MAIN
# =============================
def main():
    df_train_full = pd.read_csv(TRAIN_CSV)
    df_val_full   = pd.read_csv(VAL_CSV)

    n_train = min(5000, len(df_train_full))
    n_val   = min(500,  len(df_val_full))

    df_train = df_train_full.sample(n_train, random_state=42)
    df_val   = df_val_full.sample(n_val,   random_state=42)

    # ── Test set (held-out, never seen during training) ───────────────────
    try:
        df_test_full = pd.read_csv(TEST_CSV)
        n_test = min(1000, len(df_test_full))
    except FileNotFoundError:
        print("TEST_CSV not found — using a held-out 20% of val set for test.")
        df_test_full = df_val_full.sample(frac=0.2, random_state=99)
        n_test = len(df_test_full)
    df_test = df_test_full.sample(n_test, random_state=42)

    tokenizer    = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
    train_loader = DataLoader(MIMICDataset(df_train, tokenizer, is_train=True),
                              batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(MIMICDataset(df_val, tokenizer, is_train=False),
                              batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(MIMICDataset(df_test, tokenizer, is_train=False),
                              batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)

    model = Model().to(DEVICE)

    backbone_top_params = (
        list(model.vit_top.parameters()) +
        list(model.vit_norm.parameters())
    )
    head_params = (
        list(model.cross_v2t.parameters()) +
        list(model.cross_t2v.parameters()) +
        list(model.norm_v.parameters()) +
        list(model.norm_t.parameters()) +
        list(model.proj.parameters()) +
        list(model.proj_v.parameters()) +
        list(model.proj_t.parameters()) +
        list(model.fused_proj.parameters()) +   # ← new fused head
        [model.log_temp]
    )
    optimizer = torch.optim.AdamW([
        {"params": backbone_top_params, "lr": LR_BACKBONE, "weight_decay": 1e-4},
        {"params": head_params,         "lr": LR,          "weight_decay": 1e-4},
    ])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler    = torch.amp.GradScaler('cuda')

    best_auc      = 0.0
    best_val_loss = float("inf")
    patience      = 5
    patience_ctr  = 0

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_p   = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {total_p:,} ({100*trainable/total_p:.1f}%)")
    print(f"Backbone top LR: {LR_BACKBONE:.1e} | Head LR: {LR:.1e}")
    print(f"Train: {n_train} | Val: {n_val} | Test: {n_test} | Batch: {BATCH_SIZE}×{ACCUM_STEPS}={BATCH_SIZE*ACCUM_STEPS}eff\n")

    # ── Training history ────────────────────────────────────────────────
    history = {"train_loss": [], "val_loss": [], "val_auc": []}

    for epoch in range(1, EPOCHS + 1):
        t_loss           = train_epoch(model, train_loader, optimizer, scaler)
        torch.cuda.empty_cache()
        v_loss, auc      = validate(model, val_loader)
        torch.cuda.empty_cache()
        scheduler.step()

        history["train_loss"].append(t_loss)
        history["val_loss"].append(v_loss)
        history["val_auc"].append(auc)

        print(f"Epoch {epoch:02d}/{EPOCHS} | Train: {t_loss:.4f} | Val: {v_loss:.4f} | AUC: {auc:.4f}")

        improved = auc > best_auc or v_loss < best_val_loss
        if auc    > best_auc:      best_auc      = auc
        if v_loss < best_val_loss: best_val_loss = v_loss

        if improved:
            patience_ctr = 0
            torch.save(model.state_dict(), "radfuse_best.pt")
            print(f"  ✓ Saved (AUC={best_auc:.4f}, val_loss={best_val_loss:.4f})")
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    # ── Training curves ─────────────────────────────────────────────────
    print("\n── Generating training plots …")
    plot_loss_curves(history)
    plot_auc_curve(history)

    # ── Test evaluation ─────────────────────────────────────────────────
    print("\n── Loading best checkpoint for test evaluation …")
    model.load_state_dict(torch.load("radfuse_best.pt", map_location=DEVICE))

    per_class, macro_auc, macro_f1, preds, truths, embeddings = evaluate_test(model, test_loader)
    preds_bin = (preds >= 0.5).astype(int)

    print(f"\n{'─'*55}")
    print(f"  TEST SET  |  Macro AUC: {macro_auc:.4f}  |  Macro F1: {macro_f1:.4f}")
    print(f"{'─'*55}")
    print(f"{'Class':<30} {'AUC':>6} {'AP':>6} {'F1':>6} {'Prec':>6} {'Rec':>6} {'N':>5}")
    print(f"{'─'*55}")
    for name in CHEX_LABELS:
        m = per_class[name]
        auc_s = f"{m['auc']:.3f}" if not np.isnan(m['auc']) else "  N/A"
        ap_s  = f"{m['ap']:.3f}"  if not np.isnan(m['ap'])  else "  N/A"
        print(f"{name:<30} {auc_s:>6} {ap_s:>6} {m['f1']:>6.3f} "
              f"{m['precision']:>6.3f} {m['recall']:>6.3f} {m['support']:>5}")
    print(f"{'─'*55}\n")

    # ── Test plots ───────────────────────────────────────────────────────
    print("── Generating test evaluation plots …")
    plot_per_class_auc(per_class)
    plot_confusion_matrices(truths, preds_bin)
    plot_pr_curves(truths, preds, per_class)
    plot_metrics_summary(per_class, macro_auc, macro_f1)
    plot_tsne(embeddings)
    plot_cosine_sim_heatmap(embeddings)

    # ── Save embeddings ──────────────────────────────────────────────────
    torch.save(embeddings, "radfuse_test_embeddings.pt")

    print(f"\n{'═'*55}")
    print("  SAVED FILES")
    print(f"{'═'*55}")
    print("  radfuse_best.pt              — model weights")
    print("  radfuse_test_embeddings.pt   — CMKG-ready embeddings")
    print(f"  {PLOT_DIR}/")
    for fname in sorted(os.listdir(PLOT_DIR)):
        print(f"    {fname}")
    print(f"\n  Embedding shape  : fused_emb {embeddings['fused_emb'].shape} (L2-normalised)")
    print(f"  Best Val AUC     : {best_auc:.4f}")
    print(f"  Test Macro AUC   : {macro_auc:.4f}")
    print(f"  Test Macro F1    : {macro_f1:.4f}")
    print(f"{'═'*55}")


if __name__ == "__main__":
    main()

TEST_CSV not found — using a held-out 20% of val set for test.


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 177MB/s]


...

Epoch 20/20 | Train: 0.0426 | Val: 0.0473 | AUC: 0.9081
  ✓ Saved (AUC=0.9096, val_loss=0.0473)

── Generating training plots …
  Saved: train_val_loss_curve.png
  Saved: auc_per_epoch.png

── Loading best checkpoint for test evaluation …


Test : 100%|██████████| 25/25 [00:29<00:00,  1.17s/it]



───────────────────────────────────────────────────────
  TEST SET  |  Macro AUC: 0.8960  |  Macro F1: 0.4295
───────────────────────────────────────────────────────
Class                             AUC     AP     F1   Prec    Rec     N
───────────────────────────────────────────────────────
Enlarged Cardiomediastinum        N/A    N/A  0.000  0.000  0.000     0
Cardiomegaly                    0.908  0.687  0.476  0.714  0.357    14
Lung Opacity                      N/A    N/A  0.000  0.000  0.000     0
Lung Lesion                       N/A    N/A  0.000  0.000  0.000     0
Edema                           0.884  0.842  0.677  0.677  0.677    31
Consolidation                   0.926  0.944  0.830  0.800  0.863    51
Pneumonia                       0.905  0.903  0.779  0.833  0.732    41
Atelectasis                     0.869  0.798  0.654  0.708  0.607    28
Pneumothorax                    0.977  0.995  0.982  0.976  0.988    83
Pleural Effusion                0.993  0.999  0.982  0.97

/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


  Saved: tsne_fused_embeddings.png
  Saved: cosine_sim_heatmap.png

═══════════════════════════════════════════════════════
  SAVED FILES
═══════════════════════════════════════════════════════
  radfuse_best.pt              — model weights
  radfuse_test_embeddings.pt   — CMKG-ready embeddings
  ./radfuse_plots/
    auc_per_epoch.png
    confusion_matrix_grid.png
    cosine_sim_heatmap.png
    metrics_summary.png
    per_class_auc_bar.png
    pr_curves.png
    train_val_loss_curve.png
    tsne_fused_embeddings.png

  Embedding shape  : fused_emb torch.Size([100, 256]) (L2-normalised)
  Best Val AUC     : 0.9096
  Test Macro AUC   : 0.8960
  Test Macro F1    : 0.4295
═══════════════════════════════════════════════════════
